### Part 0: Imports & Setup

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

from scipy.io import arff
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Fixes randomness for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device detection
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"PyTorch version : {torch.__version__}")
print(f"Running on      : {device.upper()}")

PyTorch version : 2.11.0+cu130
Running on      : CPU


### Part 1: Data Loading, Preprocessing & Overview

In [8]:
data, meta = arff.loadarff('adult.arff')
df = pd.DataFrame(data)

# Identify and decode byte string columns due to ARFF loading
for col in df.columns:
    # Check if the column data type is 'object' and contains byte strings
    if df[col].dtype == 'object':
        try:
            df[col] = df[col].str.decode('utf-8')
        except AttributeError:
            # Handle cases where some 'object' columns might not be byte strings
            pass

# Replace '?' character with NaN and drop rows with missing values
df.replace('?', np.nan, inplace=True)
df.dropna(inplace=True)

### Part 2: Exploratory Data Analysis (EDA)

In [11]:
# Overview the dataset
df.info()
df.describe()
df.head(10)

X = df.drop('class', axis=1)
y = df['class']
CLASS_NAMES = y.unique()

<class 'pandas.core.frame.DataFrame'>
Index: 45222 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             45222 non-null  float64
 1   workclass       45222 non-null  object 
 2   fnlwgt          45222 non-null  float64
 3   education       45222 non-null  object 
 4   education-num   45222 non-null  float64
 5   marital-status  45222 non-null  object 
 6   occupation      45222 non-null  object 
 7   relationship    45222 non-null  object 
 8   race            45222 non-null  object 
 9   sex             45222 non-null  object 
 10  capital-gain    45222 non-null  float64
 11  capital-loss    45222 non-null  float64
 12  hours-per-week  45222 non-null  float64
 13  native-country  45222 non-null  object 
 14  class           45222 non-null  object 
dtypes: float64(6), object(9)
memory usage: 5.5+ MB


In [ ]:
# ── Basic facts ───────────────────────────────────────────────────────────────
print(f"X shape : {X.shape}  →  {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes : {list(CLASS_NAMES)}")
print()

# ── Class balance ─────────────────────────────────────────────────────────────
# This dataset is imbalanced: 75% of samples belong to class '<=50K', and 25% to '>50K'.
print("Class Distribution:")
class_counts = y.value_counts()
for class_name, count in class_counts.items():
    print(f"  Class '{class_name}' : {count} samples")
print()

# ── Feature ranges ────────────────────────────────────────────────────────────
# The features have varying ranges. For example, 'age' ranges from 17 to 90, while 'hours-per-week' ranges from 1 to 99. This suggests that feature scaling may be necessary before training a model.
print("Feature Ranges:")
for col in X.columns:
    print(f"  {col:<20} : {X[col].min():>5} to {X[col].max():>5}")

X shape : (45222, 14)  →  45222 samples, 14 features
Classes : ['<=50K', '>50K']

Class Distribution:
  Class '<=50K' : 34014 samples
  Class '>50K' : 11208 samples

Feature Ranges:
  age                  :  17.0 to  90.0
  workclass            : Federal-gov to Without-pay
  fnlwgt               : 13492.0 to 1490400.0
  education            :  10th to Some-college
  education-num        :   1.0 to  16.0
  marital-status       : Divorced to Widowed
  occupation           : Adm-clerical to Transport-moving
  relationship         : Husband to  Wife
  race                 : Amer-Indian-Eskimo to White
  sex                  : Female to  Male
  capital-gain         :   0.0 to 99999.0
  capital-loss         :   0.0 to 4356.0
  hours-per-week       :   1.0 to  99.0
  native-country       : Cambodia to Yugoslavia
